# Module 09 - 실습 3: Health Check 에이전트

## 학습 목표
- `HealthStatus` Enum과 `SystemHealth` dataclass를 정의할 수 있다
- `asyncio.gather`로 여러 시스템을 동시에 헬스체크할 수 있다
- 전체 시스템 상태를 판단하는 `run_health_check()` 함수를 구현할 수 있다

## 참조 자료
- 학습 문서: `docs/part4-production/09-external-system-integration.md` (섹션 2.4)

## 개념 요약

에이전트가 의존하는 외부 시스템의 상태를 주기적으로 확인해야 합니다.

```
Health Check 구조:

run_health_check()
    ├── check_service("redis")    ─┐
    ├── check_service("jira")     ─┼→ asyncio.gather → 전체 상태 판단
    └── check_service("gitlab")  ─┘

전체 상태:
  - 모두 HEALTHY  → HEALTHY
  - 일부 UNHEALTHY → DEGRADED
  - 모두 UNHEALTHY → UNHEALTHY
```

In [ ]:
import sys
sys.path.insert(0, '..')

import asyncio
import time
from enum import Enum
from dataclasses import dataclass

print("Health Check 실습 준비 완료")

## 실습 1: HealthStatus Enum 정의

3가지 상태를 가진 `HealthStatus` Enum을 정의하세요:
- `HEALTHY = "healthy"`
- `DEGRADED = "degraded"` (일부 시스템만 장애)
- `UNHEALTHY = "unhealthy"`

In [ ]:
# TODO: 여기에 코드를 작성하세요
# 힌트 1: class HealthStatus(Enum): 으로 시작합니다
# 힌트 2: 각 상태의 값은 소문자 문자열입니다

class HealthStatus(Enum):
    pass  # TODO: 3가지 상태 정의

In [ ]:
# 검증 셀
assert HealthStatus.HEALTHY.value == "healthy", f"HEALTHY 값이 'healthy'여야 합니다"
assert HealthStatus.DEGRADED.value == "degraded", f"DEGRADED 값이 'degraded'여야 합니다"
assert HealthStatus.UNHEALTHY.value == "unhealthy", f"UNHEALTHY 값이 'unhealthy'여야 합니다"
print("✅ 실습 1 완료! HealthStatus Enum이 올바르게 정의되었습니다.")

## 실습 2: SystemHealth dataclass 정의

개별 시스템의 상태를 나타내는 `SystemHealth` dataclass를 정의하세요:
- `name: str` - 시스템 이름
- `status: HealthStatus` - 상태
- `latency_ms: float | None = None` - 응답 지연시간 (ms)
- `error: str | None = None` - 에러 메시지

In [ ]:
# TODO: 여기에 코드를 작성하세요
# 힌트 1: @dataclass 데코레이터를 사용합니다
# 힌트 2: 기본값이 있는 필드는 기본값 없는 필드 뒤에 옵니다

@dataclass
class SystemHealth:
    """개별 시스템의 상태."""
    pass  # TODO: 4개 필드 정의

In [ ]:
# 검증 셀
health = SystemHealth(name="redis", status=HealthStatus.HEALTHY, latency_ms=12.5)
assert health.name == "redis", "name 필드가 없습니다"
assert health.status == HealthStatus.HEALTHY, "status 필드가 없습니다"
assert health.latency_ms == 12.5, "latency_ms 필드가 없습니다"
assert health.error is None, "error 기본값이 None이어야 합니다"
print("✅ 실습 2 완료! SystemHealth dataclass가 올바르게 정의되었습니다.")

## 실습 3: check_service 함수 구현

개별 서비스의 헬스체크를 수행하는 비동기 함수를 구현하세요.
실제 네트워크 호출 대신 시뮬레이션으로 구현합니다.

In [ ]:
# TODO: 여기에 코드를 작성하세요
# 힌트 1: asyncio.sleep으로 실제 네트워크 지연을 시뮬레이션합니다
# 힌트 2: 성공 시 HEALTHY, 실패 시 UNHEALTHY를 반환합니다
# 힌트 3: try/except로 예외를 처리합니다

# 서비스별 시뮬레이션 설정
SERVICE_CONFIG = {
    "redis": {"delay": 0.05, "fail": False},
    "jira": {"delay": 0.1, "fail": False},
    "gitlab": {"delay": 0.08, "fail": True},   # gitlab은 실패하도록 설정
}

async def check_service(name: str) -> SystemHealth:
    """서비스 헬스체크를 수행합니다 (시뮬레이션)."""
    config = SERVICE_CONFIG.get(name, {"delay": 0.1, "fail": False})
    
    # TODO: 헬스체크 구현
    # - asyncio.sleep(config["delay"]) 로 지연 시뮬레이션
    # - config["fail"]이 True면 UNHEALTHY 반환
    # - 그렇지 않으면 HEALTHY 반환
    pass

In [ ]:
# 단일 서비스 테스트
redis_health = await check_service("redis")
print(f"Redis: {redis_health}")

gitlab_health = await check_service("gitlab")
print(f"GitLab: {gitlab_health}")

In [ ]:
# 검증 셀 - 개별 서비스 확인
assert redis_health is not None, "check_service가 None을 반환합니다"
assert isinstance(redis_health, SystemHealth), "SystemHealth 인스턴스를 반환해야 합니다"
assert redis_health.status == HealthStatus.HEALTHY, "Redis는 HEALTHY여야 합니다"
assert redis_health.latency_ms is not None, "latency_ms가 설정되어야 합니다"
assert gitlab_health.status == HealthStatus.UNHEALTHY, "GitLab은 UNHEALTHY여야 합니다"
print("✅ 실습 3 완료! check_service가 올바르게 작동합니다.")

## 실습 4: run_health_check 함수 구현

모든 시스템의 헬스체크를 동시에 실행하고 전체 상태를 판단하는
`run_health_check()` 함수를 구현하세요.

In [ ]:
# TODO: 여기에 코드를 작성하세요
# 힌트 1: asyncio.gather로 모든 서비스를 동시에 체크합니다
# 힌트 2: UNHEALTHY 수에 따라 전체 상태 결정:
#          - 0개 UNHEALTHY → HEALTHY
#          - 일부 UNHEALTHY → DEGRADED
#          - 모두 UNHEALTHY → UNHEALTHY
# 힌트 3: 결과를 dict 형태로 반환합니다

async def run_health_check() -> dict:
    """모든 시스템의 상태를 확인합니다."""
    # TODO: 구현
    pass

In [ ]:
# 전체 헬스체크 실행
start = time.time()
health_report = await run_health_check()
elapsed = time.time() - start

import json
print(json.dumps(health_report, indent=2, ensure_ascii=False))
print(f"\n헬스체크 완료 시간: {elapsed:.2f}초")

In [ ]:
# 검증 셀
assert health_report is not None, "run_health_check가 None을 반환합니다"
assert "status" in health_report, "결과에 'status' 키가 없습니다"
assert "systems" in health_report, "결과에 'systems' 키가 없습니다"
assert len(health_report["systems"]) == 3, "3개 시스템을 체크해야 합니다"
# gitlab이 실패하므로 전체 상태는 DEGRADED여야 함
assert health_report["status"] == HealthStatus.DEGRADED.value, \
    f"일부 실패 시 DEGRADED여야 합니다. 현재: {health_report['status']}"
# 동시 실행이므로 약 0.1초 이내여야 함
assert elapsed < 0.5, f"동시 실행이 너무 오래 걸립니다: {elapsed:.2f}초"
print(f"✅ 실습 4 완료! run_health_check가 올바르게 동작합니다. (전체 상태: {health_report['status']})")

## 정리

이번 실습에서 배운 내용:
1. `Enum`으로 상태를 명확하게 표현할 수 있다
2. `dataclass`로 구조화된 데이터를 정의할 수 있다
3. `asyncio.gather`로 여러 시스템을 동시에 체크하여 빠르게 헬스체크할 수 있다
4. 개별 결과를 집계하여 전체 시스템 상태를 판단할 수 있다

## 다음 모듈
- **Module 10**: `module_10_resource_optimization/` - 리소스 최적화